# Construction Workplace Productivity — Exploratory Data Analysis

This notebook profiles, cleans, and explores the workbook in `Data/construction_workplace_productivity_pack.xlsx`. It examines productivity against plan, time and project subgroups, labour and cost, work conditions, workforce, safety, rework, and quality. Cleaning is explicit and non-destructive: the source frame is preserved, questionable records are flagged, and no outcome is imputed or outlier clipped.

> **Important context:** The workbook describes a fictional, synthetic learning dataset. Observed associations are descriptive, not causal. `Delay_Risk_Score` is illustrative and is not a validated risk or prediction measure.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
RANDOM_SEED = 42
print('pandas', pd.__version__, '| numpy', np.__version__, '| seaborn', sns.__version__)

## 1. Load the workbook and reference sheets

The path is resolved from the notebook's working directory, with a parent-directory fallback for common VS Code launch locations. All workbook sheets are loaded so the dictionary and interpretation notes can be consulted alongside the observations.

In [ ]:
candidates = [
    Path.cwd() / 'Data' / 'construction_workplace_productivity_pack.xlsx',
    Path.cwd().parent / 'Data' / 'construction_workplace_productivity_pack.xlsx',
]
WORKBOOK_PATH = next((path.resolve() for path in candidates if path.exists()), None)
if WORKBOOK_PATH is None:
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find the source workbook. Searched:\n{searched}')

sheets = pd.read_excel(WORKBOOK_PATH, sheet_name=None, engine='openpyxl')
print('Workbook:', WORKBOOK_PATH)
print('Sheets:', {name: frame.shape for name, frame in sheets.items()})
raw_df = sheets['Productivity_Data'].copy(deep=True)
data_dictionary = sheets.get('Data_Dictionary', pd.DataFrame())
read_me = sheets.get('Read_Me', pd.DataFrame())
interpretation_notes = sheets.get('Interpretation_Notes', pd.DataFrame())
display(read_me)
display(interpretation_notes)

## 2. Initial profile and source metadata

Start with the workbook's own definitions and raw shape/types before transforming anything. The source distinguishes productivity ratios (0–1) from percentage-point fields.

In [ ]:
print(f'Raw observations: {len(raw_df):,} | raw columns: {raw_df.shape[1]}')
display(data_dictionary)
display(raw_df.head())
display(pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing': raw_df.isna().sum(),
    'missing_pct': 100 * raw_df.isna().mean(),
    'unique_non_null': raw_df.nunique(dropna=True),
}).sort_values(['missing', 'unique_non_null'], ascending=[False, True]))

## 3. Data cleaning and quality checks

### Cleaning policy
- Keep `raw_df` unchanged and work on a copy.
- Trim whitespace in column names and text values; turn empty text into a missing value. Do not convert meaningful labels such as `None recorded` into nulls.
- Parse dates and numeric measures explicitly, recording any values that fail conversion.
- Flag duplicate IDs/rows, impossible bounds, and productivity-variance inconsistencies. Do not drop flagged records, impute productivity outcomes, or clip potential outliers.
- Keep `cleaned_df` row-preserving; later analyses exclude only rows missing the specific metric being calculated.

In [ ]:
cleaned_df = raw_df.copy(deep=True)
cleaned_df.columns = cleaned_df.columns.astype(str).str.strip()

text_columns = cleaned_df.select_dtypes(include=['object', 'string']).columns.tolist()
blank_text_cells = 0
for column in text_columns:
    values = cleaned_df[column].astype('string').str.strip()
    blank_text_cells += int(values.eq('').sum())
    cleaned_df[column] = values.replace('', pd.NA)

date_columns = [column for column in ['Observation_Date'] if column in cleaned_df.columns]
date_parse_failures = {}
for column in date_columns:
    before = cleaned_df[column].notna()
    cleaned_df[column] = pd.to_datetime(cleaned_df[column], errors='coerce')
    date_parse_failures[column] = int((before & cleaned_df[column].isna()).sum())

numeric_columns = [
    'Crew_Size', 'Planned_Productivity', 'Actual_Productivity', 'Productivity_Variance',
    'Planned_Labour_Hours', 'Overtime_Hours', 'Estimated_Labour_Cost_GBP', 'Incentive_GBP',
    'Idle_Time_Hours', 'Idle_Workers', 'Standard_Labour_Minutes_Per_Unit',
    'Work_In_Progress_Units', 'Planned_Quantity', 'Completed_Quantity', 'Progress_Percent',
    'Material_Availability_Percent', 'Equipment_Availability_Percent', 'Equipment_Downtime_Hours',
    'Temperature_C', 'Rainfall_mm', 'Wind_Speed_kmh', 'Safety_Incidents',
    'Safety_Standdown_Hours', 'Rework_Hours', 'Rework_Rate_Percent',
    'Workforce_Absence_Rate_Percent', 'Average_Workforce_Experience_Years', 'Training_Hours',
    'Design_Change_Count', 'Supervisor_Experience_Years', 'Quality_Pass_Rate_Percent',
    'Delay_Risk_Score',
]
numeric_parse_failures = {}
for column in numeric_columns:
    if column in cleaned_df.columns:
        before = cleaned_df[column].notna()
        cleaned_df[column] = pd.to_numeric(cleaned_df[column], errors='coerce')
        numeric_parse_failures[column] = int((before & cleaned_df[column].isna()).sum())

cleaned_df['_QC_duplicate_record_id'] = (
    cleaned_df['Record_ID'].duplicated(keep=False)
    if 'Record_ID' in cleaned_df.columns else False
)
cleaned_df['_QC_exact_duplicate_row'] = cleaned_df.duplicated(keep=False)

range_flags = pd.DataFrame(index=cleaned_df.index)
for column in ['Planned_Productivity', 'Actual_Productivity']:
    if column in cleaned_df.columns:
        range_flags[f'{column}_outside_0_1'] = ~cleaned_df[column].between(0, 1, inclusive='both') & cleaned_df[column].notna()
for column in [
    'Material_Availability_Percent', 'Equipment_Availability_Percent',
    'Rework_Rate_Percent', 'Workforce_Absence_Rate_Percent', 'Quality_Pass_Rate_Percent',
]:
    if column in cleaned_df.columns:
        range_flags[f'{column}_outside_0_100'] = ~cleaned_df[column].between(0, 100, inclusive='both') & cleaned_df[column].notna()
if 'Delay_Risk_Score' in cleaned_df.columns:
    range_flags['Delay_Risk_Score_outside_0_100'] = ~cleaned_df['Delay_Risk_Score'].between(0, 100, inclusive='both') & cleaned_df['Delay_Risk_Score'].notna()

nonnegative_columns = [
    'Crew_Size', 'Planned_Labour_Hours', 'Overtime_Hours', 'Estimated_Labour_Cost_GBP',
    'Incentive_GBP', 'Idle_Time_Hours', 'Idle_Workers', 'Standard_Labour_Minutes_Per_Unit',
    'Work_In_Progress_Units', 'Planned_Quantity', 'Completed_Quantity', 'Equipment_Downtime_Hours',
    'Rainfall_mm', 'Wind_Speed_kmh', 'Safety_Incidents', 'Safety_Standdown_Hours', 'Rework_Hours',
    'Workforce_Absence_Rate_Percent', 'Average_Workforce_Experience_Years', 'Training_Hours',
    'Design_Change_Count', 'Supervisor_Experience_Years',
]
for column in nonnegative_columns:
    if column in cleaned_df.columns:
        range_flags[f'{column}_negative'] = cleaned_df[column].lt(0) & cleaned_df[column].notna()

for column in range_flags.columns:
    cleaned_df[f'_QC_{column}'] = range_flags[column].fillna(False).astype(bool)

if {'Actual_Productivity', 'Planned_Productivity', 'Productivity_Variance'}.issubset(cleaned_df.columns):
    expected_variance = cleaned_df['Actual_Productivity'] - cleaned_df['Planned_Productivity']
    mismatch = ~np.isclose(cleaned_df['Productivity_Variance'], expected_variance, atol=0.001, rtol=0.0, equal_nan=True)
    cleaned_df['_QC_productivity_variance_mismatch'] = mismatch & cleaned_df['Productivity_Variance'].notna() & expected_variance.notna()
else:
    cleaned_df['_QC_productivity_variance_mismatch'] = False

qc_columns = [column for column in cleaned_df.columns if column.startswith('_QC_')]
cleaned_df['_QC_flag_count'] = cleaned_df[qc_columns].sum(axis=1).astype(int)
print(f'Row-preserving clean frame: {len(cleaned_df):,} rows (raw: {len(raw_df):,})')
print(f'Trimmed blank text cells converted to missing: {blank_text_cells:,}')
print('Date parse failures:', date_parse_failures)
print('Numeric parse failures:', {k: v for k, v in numeric_parse_failures.items() if v})

In [ ]:
CLEANED_EXPORT_PATH = WORKBOOK_PATH.with_name(f'{WORKBOOK_PATH.stem}_cleaned.xlsx')
cleaned_export_df = cleaned_df.loc[:, ~cleaned_df.columns.str.startswith('_QC_')]
cleaned_export_df.to_excel(CLEANED_EXPORT_PATH, index=False, engine='openpyxl')
print(f'Exported cleaned data without QC columns to {CLEANED_EXPORT_PATH}')

In [ ]:
audit = pd.DataFrame([
    ('Rows in source', len(raw_df)),
    ('Rows after cleaning (no rows dropped)', len(cleaned_df)),
    ('Exact duplicate rows (rows implicated)', int(cleaned_df['_QC_exact_duplicate_row'].sum())),
    ('Duplicate Record_IDs (rows implicated)', int(cleaned_df['_QC_duplicate_record_id'].sum())),
    ('Rows with at least one quality flag', int(cleaned_df['_QC_flag_count'].gt(0).sum())),
    ('Total missing cells, source', int(raw_df.isna().sum().sum())),
    ('Total missing cells, cleaned', int(cleaned_df[raw_df.columns].isna().sum().sum())),
], columns=['Check', 'Count'])
display(audit)
flag_counts = cleaned_df[qc_columns].sum().sort_values(ascending=False).rename('flagged_rows').to_frame()
display(flag_counts[flag_counts['flagged_rows'].gt(0)])

missing_profile = pd.DataFrame({
    'missing': cleaned_df[raw_df.columns].isna().sum(),
    'missing_pct': 100 * cleaned_df[raw_df.columns].isna().mean(),
}).query('missing > 0').sort_values('missing', ascending=False)
display(missing_profile)
if not missing_profile.empty:
    ax = missing_profile['missing_pct'].sort_values().plot.barh(figsize=(9, max(3, 0.28 * len(missing_profile))), color='#3973ac')
    ax.set(title='Missing values by field', xlabel='Missing (%)', ylabel='')
    plt.tight_layout()
    plt.show()

### Outlier review (not automatic removal)

The IQR screen below is a triage aid, not a declaration that a value is wrong. Construction observations can be skewed and unusual values may be genuine. Review distributions and source context before any exclusion.

In [ ]:
numeric_data_columns = [column for column in numeric_columns if column in cleaned_df.columns]
outlier_rows = []
for column in numeric_data_columns:
    values = cleaned_df[column].dropna()
    if len(values) < 4:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        'field': column, 'non_missing': len(values), 'q1': q1, 'q3': q3,
        'iqr_lower_fence': lower, 'iqr_upper_fence': upper,
        'outside_fences': int(((values < lower) | (values > upper)).sum()),
        'outside_fences_pct': 100 * ((values < lower) | (values > upper)).mean(),
    })
outlier_review = pd.DataFrame(outlier_rows).sort_values('outside_fences_pct', ascending=False)
display(outlier_review.head(20))

## 4. Descriptive statistics and categories

Summaries use the cleaned values without replacing missing data. Means are paired with medians and quartiles because operational measures may be skewed.

In [ ]:
display(cleaned_df[numeric_data_columns].describe(percentiles=[.05, .25, .5, .75, .95]).T)
categorical_columns = [column for column in [
    'Quarter', 'Day_of_Week', 'Site_Region', 'Project_Type', 'Project_Phase',
    'Work_Package', 'Work_Package_Complexity', 'Quantity_Unit', 'Weather_Conditions',
    'Site_Constraints', 'Site_Access_Condition',
] if column in cleaned_df.columns]
category_profile = pd.DataFrame([
    {'field': column, 'unique_non_null': cleaned_df[column].nunique(dropna=True),
     'missing': int(cleaned_df[column].isna().sum()),
     'most_common': cleaned_df[column].mode(dropna=True).iloc[0] if not cleaned_df[column].mode(dropna=True).empty else pd.NA,
     'most_common_n': int(cleaned_df[column].value_counts(dropna=True).iloc[0]) if not cleaned_df[column].value_counts(dropna=True).empty else 0}
    for column in categorical_columns
])
display(category_profile)
for column in categorical_columns:
    print(f'\n{column} — top values')
    display(cleaned_df[column].value_counts(dropna=False).head(12).rename('observations').to_frame())

## 5. Productivity against plan

`Planned_Productivity` and `Actual_Productivity` are ratios from 0 to 1; multiply by 100 for percentages. `Productivity_Variance` is actual minus planned, so its sign indicates whether the observation is above or below plan. Use paired, valid outcome values only.

In [ ]:
required_outcomes = ['Planned_Productivity', 'Actual_Productivity', 'Productivity_Variance']
available_outcomes = [column for column in required_outcomes if column in cleaned_df.columns]
if len(available_outcomes) == len(required_outcomes):
    outcomes = cleaned_df.dropna(subset=required_outcomes).copy()
    outcomes = outcomes[
        outcomes['Planned_Productivity'].between(0, 1)
        & outcomes['Actual_Productivity'].between(0, 1)
    ].copy()
    outcomes['Actual_Productivity_Pct'] = 100 * outcomes['Actual_Productivity']
    outcomes['Planned_Productivity_Pct'] = 100 * outcomes['Planned_Productivity']
    outcomes['Variance_Pct_Points'] = 100 * outcomes['Productivity_Variance']
    outcomes['Above_Plan'] = outcomes['Productivity_Variance'] > 0
    outcomes['At_Plan'] = np.isclose(outcomes['Productivity_Variance'], 0, atol=0.001)
    display(pd.DataFrame({
        'metric': ['Valid paired observations', 'Mean planned productivity (%)', 'Mean actual productivity (%)',
                   'Median actual productivity (%)', 'Mean variance (percentage points)',
                   'Median variance (percentage points)', 'Share above plan (%)', 'Share at plan (%)'],
        'value': [len(outcomes), outcomes['Planned_Productivity_Pct'].mean(), outcomes['Actual_Productivity_Pct'].mean(),
                  outcomes['Actual_Productivity_Pct'].median(), outcomes['Variance_Pct_Points'].mean(),
                  outcomes['Variance_Pct_Points'].median(), 100 * outcomes['Above_Plan'].mean(),
                  100 * outcomes['At_Plan'].mean()],
    }))
else:
    warnings.warn(f'Outcome analysis skipped; missing columns: {sorted(set(required_outcomes) - set(cleaned_df.columns))}')

In [ ]:
if 'outcomes' in globals() and not outcomes.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    sns.histplot(outcomes['Actual_Productivity_Pct'], bins=30, kde=True, ax=axes[0], color='#3973ac')
    axes[0].axvline(outcomes['Planned_Productivity_Pct'].mean(), color='#d1495b', linestyle='--', label='Mean planned')
    axes[0].set(title='Actual productivity', xlabel='Productivity (%)')
    axes[0].legend()
    sns.histplot(outcomes['Variance_Pct_Points'], bins=30, kde=True, ax=axes[1], color='#4c956c')
    axes[1].axvline(0, color='black', linestyle='--')
    axes[1].set(title='Actual minus planned', xlabel='Variance (percentage points)')
    sns.scatterplot(data=outcomes, x='Planned_Productivity_Pct', y='Actual_Productivity_Pct', alpha=.45, ax=axes[2])
    axes[2].plot([0, 100], [0, 100], color='#d1495b', linestyle='--', label='Actual = plan')
    axes[2].set(title='Actual vs planned', xlabel='Planned (%)', ylabel='Actual (%)')
    axes[2].legend()
    plt.tight_layout()
    plt.show()

## 6. Performance by project and work context

Compare means, medians, sample sizes, and the share above plan. The minimum-group-size filter is configurable and shown so small groups are not over-interpreted. These are descriptive comparisons, not adjusted estimates.

In [ ]:
def productivity_by(group_column, data=None, min_n=10):
    """Summarize productivity for sufficiently observed groups."""
    data = outcomes if data is None else data
    if group_column not in data.columns:
        return pd.DataFrame()
    result = data.groupby(group_column, dropna=False).agg(
        observations=('Actual_Productivity', 'count'),
        mean_actual=('Actual_Productivity', 'mean'),
        median_actual=('Actual_Productivity', 'median'),
        mean_variance=('Productivity_Variance', 'mean'),
        median_variance=('Productivity_Variance', 'median'),
        share_above_plan=('Above_Plan', 'mean'),
    )
    result = result[result['observations'] >= min_n].sort_values('mean_variance', ascending=False)
    result['mean_actual_pct'] = 100 * result['mean_actual']
    result['mean_variance_pp'] = 100 * result['mean_variance']
    result['share_above_plan_pct'] = 100 * result['share_above_plan']
    return result

for group_column in ['Site_Region', 'Project_Type', 'Project_Phase', 'Work_Package', 'Work_Package_Complexity', 'Site_Access_Condition']:
    if group_column in outcomes.columns:
        print(f'\n{group_column} (groups with at least 10 observations)')
        display(productivity_by(group_column))

In [ ]:
plot_groups = ['Site_Region', 'Project_Type', 'Project_Phase', 'Work_Package_Complexity', 'Site_Access_Condition']
for group_column in plot_groups:
    if group_column not in outcomes.columns:
        continue
    summary = productivity_by(group_column)
    if summary.empty:
        continue
    summary = summary.sort_values('mean_variance_pp')
    fig, ax = plt.subplots(figsize=(9, max(3.2, 0.38 * len(summary))))
    ax.barh(summary.index.astype(str), summary['mean_variance_pp'], color=np.where(summary['mean_variance_pp'] >= 0, '#4c956c', '#d1495b'))
    ax.axvline(0, color='black', linewidth=1)
    ax.set(title=f'Mean productivity variance by {group_column}', xlabel='Mean actual − planned (percentage points)', ylabel='')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'Crew_Size' in outcomes.columns:
    outcomes['Crew_Size_Band'] = pd.cut(
        outcomes['Crew_Size'], bins=[0, 10, 20, 35, 50, np.inf],
        labels=['1–10', '11–20', '21–35', '36–50', '51+'], include_lowest=True,
    )
    display(productivity_by('Crew_Size_Band'))
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.boxplot(data=outcomes, x='Crew_Size_Band', y='Variance_Pct_Points', showfliers=False, ax=ax)
    ax.axhline(0, color='#d1495b', linestyle='--')
    ax.set(title='Productivity variance by crew-size band', xlabel='Crew size', ylabel='Actual − planned (pp)')
    plt.tight_layout()
    plt.show()

## 7. Time patterns

Daily records may cluster on the same dates; the monthly series therefore reports the number of observations as well as mean productivity. A rolling average smooths visual noise but does not establish a trend mechanism.

In [ ]:
if 'Observation_Date' in outcomes.columns:
    time_data = outcomes.dropna(subset=['Observation_Date']).copy()
    if not time_data.empty:
        time_data['Month'] = time_data['Observation_Date'].dt.to_period('M').dt.to_timestamp()
        monthly = time_data.groupby('Month').agg(
            observations=('Actual_Productivity', 'count'),
            mean_actual=('Actual_Productivity', 'mean'),
            mean_planned=('Planned_Productivity', 'mean'),
            mean_variance=('Productivity_Variance', 'mean'),
        ).reset_index()
        display(monthly.head(12))
        display(monthly.tail(12))
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(monthly['Month'], 100 * monthly['mean_actual'], marker='.', label='Actual')
        ax.plot(monthly['Month'], 100 * monthly['mean_planned'], linestyle='--', label='Planned')
        ax.set(title='Monthly mean productivity', xlabel='Month', ylabel='Productivity (%)')
        ax.legend()
        plt.tight_layout()
        plt.show()
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.bar(monthly['Month'], 100 * monthly['mean_variance'], width=20, color=np.where(monthly['mean_variance'] >= 0, '#4c956c', '#d1495b'))
        ax.axhline(0, color='black', linewidth=1)
        ax.set(title='Monthly mean variance from plan', xlabel='Month', ylabel='Variance (percentage points)')
        plt.tight_layout()
        plt.show()
        print('Months represented:', len(monthly), '| date range:', time_data['Observation_Date'].min().date(), 'to', time_data['Observation_Date'].max().date())

## 8. Labour, overtime, idle time, and cost

Use direct measures and clearly labelled derived ratios. Cost per completed quantity is only comparable within a common unit and work context, so it is not ranked across unlike `Quantity_Unit` values.

In [ ]:
operations = cleaned_df.copy()
if {'Estimated_Labour_Cost_GBP', 'Completed_Quantity'}.issubset(operations.columns):
    denominator = operations['Completed_Quantity'].where(operations['Completed_Quantity'] > 0)
    operations['Cost_per_Completed_Quantity_GBP'] = operations['Estimated_Labour_Cost_GBP'] / denominator
if {'Overtime_Hours', 'Planned_Labour_Hours'}.issubset(operations.columns):
    denominator = operations['Planned_Labour_Hours'].where(operations['Planned_Labour_Hours'] > 0)
    operations['Overtime_to_Planned_Labour_Ratio'] = operations['Overtime_Hours'] / denominator

operation_fields = [column for column in [
    'Planned_Labour_Hours', 'Overtime_Hours', 'Overtime_to_Planned_Labour_Ratio',
    'Estimated_Labour_Cost_GBP', 'Incentive_GBP', 'Idle_Time_Hours', 'Idle_Workers',
    'Equipment_Downtime_Hours', 'Work_In_Progress_Units', 'Cost_per_Completed_Quantity_GBP',
] if column in operations.columns]
display(operations[operation_fields].describe(percentiles=[.5, .9, .95]).T)

if 'outcomes' in globals():
    joined_ops = outcomes.join(operations[operation_fields], how='left', rsuffix='_ops')
    for measure in ['Overtime_Hours', 'Idle_Time_Hours', 'Equipment_Downtime_Hours', 'Planned_Labour_Hours', 'Estimated_Labour_Cost_GBP']:
        if measure in joined_ops.columns:
            fig, ax = plt.subplots(figsize=(6, 4))
            sns.regplot(data=joined_ops, x=measure, y='Variance_Pct_Points', lowess=True,
                        scatter_kws={'alpha': .3, 's': 18}, line_kws={'color': '#d1495b'}, ax=ax)
            ax.set(title=f'{measure} and productivity variance (descriptive)', ylabel='Actual − planned (pp)')
            plt.tight_layout()
            plt.show()

## 9. Materials, equipment, weather, and site constraints

These panels compare distributions across observed categories and quantities. A site may face several simultaneous conditions; the data do not isolate their independent effects.

In [ ]:
for measure in ['Material_Availability_Percent', 'Equipment_Availability_Percent', 'Equipment_Downtime_Hours', 'Rainfall_mm', 'Wind_Speed_kmh', 'Temperature_C']:
    if measure in cleaned_df.columns:
        fig, ax = plt.subplots(figsize=(7, 3.8))
        sns.histplot(cleaned_df[measure].dropna(), bins=30, kde=True, ax=ax, color='#3973ac')
        ax.set(title=f'Distribution: {measure}', xlabel=measure)
        plt.tight_layout()
        plt.show()

for group_column in ['Weather_Conditions', 'Site_Constraints', 'Site_Access_Condition']:
    if group_column in outcomes.columns:
        summary = productivity_by(group_column)
        print(f'\nProductivity by {group_column}')
        display(summary)
        if not summary.empty:
            plot_data = outcomes[outcomes[group_column].isin(summary.index)]
            fig, ax = plt.subplots(figsize=(9, max(3.2, 0.36 * summary.shape[0])))
            order = summary.sort_values('mean_variance').index
            sns.boxplot(data=plot_data, y=group_column, x='Variance_Pct_Points', order=order, showfliers=False, ax=ax)
            ax.axvline(0, color='#d1495b', linestyle='--')
            ax.set(title=f'Productivity variance by {group_column}', xlabel='Actual − planned (pp)', ylabel='')
            plt.tight_layout()
            plt.show()

for measure in ['Material_Availability_Percent', 'Equipment_Availability_Percent', 'Equipment_Downtime_Hours', 'Rainfall_mm', 'Wind_Speed_kmh', 'Temperature_C']:
    if measure in outcomes.columns:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.regplot(data=outcomes, x=measure, y='Variance_Pct_Points', lowess=True,
                    scatter_kws={'alpha': .3, 's': 18}, line_kws={'color': '#d1495b'}, ax=ax)
        ax.set(title=f'{measure} and productivity variance (descriptive)', ylabel='Actual − planned (pp)')
        plt.tight_layout()
        plt.show()

## 10. Workforce, safety, rework, and quality

Review the outcome alongside worker experience, absence, training, safety incidents, rework, and quality pass rate. Safety events and quality measures are especially important context and should not be collapsed into productivity alone.

In [ ]:
people_quality_fields = [column for column in [
    'Average_Workforce_Experience_Years', 'Supervisor_Experience_Years', 'Workforce_Absence_Rate_Percent',
    'Training_Hours', 'Safety_Incidents', 'Safety_Standdown_Hours', 'Rework_Hours',
    'Rework_Rate_Percent', 'Quality_Pass_Rate_Percent', 'Design_Change_Count',
] if column in cleaned_df.columns]
display(cleaned_df[people_quality_fields].describe(percentiles=[.5, .9, .95]).T)

for measure in people_quality_fields:
    if measure in outcomes.columns:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.regplot(data=outcomes, x=measure, y='Variance_Pct_Points', lowess=True,
                    scatter_kws={'alpha': .3, 's': 18}, line_kws={'color': '#d1495b'}, ax=ax)
        ax.set(title=f'{measure} and productivity variance (descriptive)', ylabel='Actual − planned (pp)')
        plt.tight_layout()
        plt.show()

if {'Safety_Incidents', 'Quality_Pass_Rate_Percent', 'Rework_Rate_Percent'}.issubset(cleaned_df.columns):
    display(cleaned_df.groupby('Safety_Incidents', dropna=False).agg(
        observations=('Record_ID', 'count') if 'Record_ID' in cleaned_df.columns else ('Safety_Incidents', 'size'),
        mean_quality_pass=('Quality_Pass_Rate_Percent', 'mean'),
        median_rework_rate=('Rework_Rate_Percent', 'median'),
        mean_rework_rate=('Rework_Rate_Percent', 'mean'),
    ).sort_index())

## 11. Associations and illustrative delay score

Spearman rank correlations are used to describe monotonic association and reduce sensitivity to scale/outliers. Correlation is not causation, groups/records may not be independent, and the synthetic nature limits external validity. The delay score is shown only as an illustrative field, never as ground truth or a validated model target.

In [ ]:
correlation_candidates = [column for column in [
    'Actual_Productivity', 'Planned_Productivity', 'Productivity_Variance',
    'Crew_Size', 'Planned_Labour_Hours', 'Overtime_Hours', 'Estimated_Labour_Cost_GBP',
    'Idle_Time_Hours', 'Material_Availability_Percent', 'Equipment_Availability_Percent',
    'Equipment_Downtime_Hours', 'Temperature_C', 'Rainfall_mm', 'Wind_Speed_kmh',
    'Safety_Incidents', 'Safety_Standdown_Hours', 'Rework_Hours', 'Rework_Rate_Percent',
    'Workforce_Absence_Rate_Percent', 'Average_Workforce_Experience_Years', 'Training_Hours',
    'Design_Change_Count', 'Supervisor_Experience_Years', 'Quality_Pass_Rate_Percent',
    'Delay_Risk_Score',
] if column in cleaned_df.columns]
spearman = cleaned_df[correlation_candidates].corr(method='spearman', min_periods=20)
if not spearman.empty:
    fig, ax = plt.subplots(figsize=(13, 10))
    sns.heatmap(spearman, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title('Spearman correlations among selected numeric fields')
    plt.tight_layout()
    plt.show()

target = 'Productivity_Variance'
if target in spearman.columns:
    target_associations = spearman[target].drop(labels=[target], errors='ignore').dropna().sort_values(key=lambda series: series.abs(), ascending=False)
    display(target_associations.rename('spearman_rho_with_productivity_variance').to_frame())
if 'Delay_Risk_Score' in spearman.columns:
    display(spearman['Delay_Risk_Score'].drop(labels=['Delay_Risk_Score'], errors='ignore').sort_values(key=lambda series: series.abs(), ascending=False).rename('spearman_rho_with_illustrative_delay_score').to_frame())

## 12. Evidence-led takeaways and limitations

The cell below extracts a few descriptive extremes directly from this workbook rather than hard-coding conclusions. Treat them as prompts for investigation, not causal recommendations. Counts are included to make coverage visible.

In [ ]:
print(f'Analysis includes {len(outcomes):,} valid paired productivity observations from {len(cleaned_df):,} source rows.')
print(f"Overall mean actual productivity: {100 * outcomes['Actual_Productivity'].mean():.1f}% versus "
      f"{100 * outcomes['Planned_Productivity'].mean():.1f}% planned; mean variance "
      f"{100 * outcomes['Productivity_Variance'].mean():+.2f} percentage points.")
print(f"{100 * outcomes['Above_Plan'].mean():.1f}% of paired observations are above plan.")

for group_column in ['Site_Region', 'Project_Type', 'Project_Phase', 'Work_Package_Complexity', 'Site_Access_Condition']:
    if group_column not in outcomes.columns:
        continue
    summary = productivity_by(group_column, min_n=10)
    if len(summary) >= 2:
        low = summary.sort_values('mean_variance').iloc[0]
        high = summary.sort_values('mean_variance').iloc[-1]
        print(f"{group_column}: lowest mean variance {summary.index[summary.index.get_loc(low.name)]!r} "
              f"({100 * low['mean_variance']:+.2f} pp, n={int(low['observations'])}); "
              f"highest {high.name!r} ({100 * high['mean_variance']:+.2f} pp, n={int(high['observations'])}).")

print('\nInterpretation guardrails:')
print('- Synthetic observations are not evidence about real construction projects or workers.')
print('- Group differences and correlations do not control for confounding, project mix, repeated crews, or time effects.')
print('- IQR/range flags identify records to review; they are not automatic grounds for deletion.')
print('- Delay_Risk_Score is illustrative only; this notebook does not validate or train a risk model.')
print('- Before operational decisions, validate definitions, provenance, coverage, and patterns against real source data.')

## Reproducibility and next steps

Run all cells from top to bottom. If the Python kernel does not have the required packages, install `pandas`, `openpyxl`, `numpy`, `matplotlib`, and `seaborn` into that kernel. Keep the workbook under the relative `Data/` folder (or adjust `WORKBOOK_PATH`). For follow-up work, consider within-project/crew comparisons, time-aware models, and sensitivity checks only after validating the synthetic feature-generation process and the meaning of each metric.